# Sanity Check: Are the request headers mandatory?

The scraper (`scrape_characters.ipynb`) uses a full browser `User-Agent` header, with the comment *“the wiki returns 403 for generic bot strings”*. This notebook verifies that claim empirically by hitting the same endpoints under four header configurations and comparing status codes + response sizes.

Configurations tested:
1. **No headers** — `requests` default (sends `User-Agent: python-requests/<version>`)
2. **Empty User-Agent** — explicitly blanked out
3. **Generic bot string** — `User-Agent: bot`
4. **Full browser headers** — the ones used in the real scraper

In [1]:
import requests

BASE = 'https://awoiaf.westeros.org'
PREFIX = '/index.php/'

# A few representative endpoints: the list page and a handful of character pages.   
ENDPOINTS = [
    'List_of_characters',
    'Eddard_Stark',
    'Jon_Snow',
    'Tyrion_Lannister',
    'Daenerys_Targaryen',
]

SCRAPER_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

CONFIGS = {
    'no_headers':      None,
    'empty_UA':        {'User-Agent': ''},
    'generic_bot':     {'User-Agent': 'bot'},
    'browser_headers': SCRAPER_HEADERS,
}

## Probe each endpoint under each header configuration
Uses a **fresh `requests.Session` per config** so nothing leaks between runs (no cached cookies, no persisted headers).

In [2]:
def probe(endpoint, headers):
    s = requests.Session()
    if headers is not None:
        s.headers.update(headers)
    try:
        r = s.get(BASE + PREFIX + endpoint, timeout=20)
        return {
            'status': r.status_code,
            'bytes': len(r.content),
            'ok': r.status_code == 200,
        }
    except requests.RequestException as e:
        return {'status': 'ERR', 'bytes': 0, 'ok': False, 'error': str(e)}


results = {cfg: {} for cfg in CONFIGS}
for cfg, headers in CONFIGS.items():
    print(f'--- {cfg} ---')
    for ep in ENDPOINTS:
        res = probe(ep, headers)
        results[cfg][ep] = res
        print(f'  {ep:22s} status={res["status"]}  bytes={res["bytes"]}')
    print()

--- no_headers ---
  List_of_characters     status=403  bytes=5453
  Eddard_Stark           status=403  bytes=5413
  Jon_Snow               status=403  bytes=5401
  Tyrion_Lannister       status=403  bytes=5425
  Daenerys_Targaryen     status=403  bytes=5453

--- empty_UA ---
  List_of_characters     status=403  bytes=5389
  Eddard_Stark           status=403  bytes=5371
  Jon_Snow               status=403  bytes=5359
  Tyrion_Lannister       status=403  bytes=5383
  Daenerys_Targaryen     status=403  bytes=5389

--- generic_bot ---
  List_of_characters     status=200  bytes=422692
  Eddard_Stark           status=200  bytes=319070
  Jon_Snow               status=200  bytes=312840
  Tyrion_Lannister       status=200  bytes=273240
  Daenerys_Targaryen     status=200  bytes=283091

--- browser_headers ---
  List_of_characters     status=200  bytes=423190
  Eddard_Stark           status=200  bytes=319568
  Jon_Snow               status=200  bytes=313338
  Tyrion_Lannister       status=200  

## Summary table

In [3]:
import pandas as pd

summary = pd.DataFrame(
    {cfg: {ep: results[cfg][ep]['status'] for ep in ENDPOINTS} for cfg in CONFIGS}
)
print('Status codes by (endpoint, config):')
print(summary)

success_rate = {
    cfg: sum(1 for ep in ENDPOINTS if results[cfg][ep]['ok']) / len(ENDPOINTS)
    for cfg in CONFIGS
}
print('\nSuccess rate per config:')
for cfg, rate in success_rate.items():
    print(f'  {cfg:17s} {rate:.0%}')

Status codes by (endpoint, config):
                    no_headers  empty_UA  generic_bot  browser_headers
List_of_characters         403       403          200              200
Eddard_Stark               403       403          200              200
Jon_Snow                   403       403          200              200
Tyrion_Lannister           403       403          200              200
Daenerys_Targaryen         403       403          200              200

Success rate per config:
  no_headers        0%
  empty_UA          0%
  generic_bot       100%
  browser_headers   100%


## Verdict
Read the summary above:

- If `no_headers` / `empty_UA` / `generic_bot` return **403 (Forbidden)** while `browser_headers` returns **200**, the custom `User-Agent` is **mandatory** — the wiki blocks generic `python-requests/...` bot strings.
- If all four configurations return **200**, the header is **not** required and the scraper could be simplified.

Re-run this notebook if the wiki’s bot policy changes.